**Author:** Ashutosh Jayant  
**Project:** India State Competitiveness Index (ISCI)

# 12 - Development Priorities

**This notebook answers one research question.**

**Research Question:** Where should each state focus to improve?

This notebook does not recommend specific government schemes or policy mechanisms. It
identifies the areas where the data suggests that improvement would matter most. These are
evidence-based priorities from the measured gaps, not policy prescriptions.

## Setup

Load the ranking, the state groups, and the gaps to the top group. Also set a simple map
from each number to a broad area (like industry, education or power).

In [5]:
import sys
from pathlib import Path
import pandas as pd

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

reports_dir = root / "version2" / "reports"

index = pd.read_csv(root / "results" / "competitiveness_index.csv", index_col=0)
clusters = pd.read_csv(reports_dir / "state_clusters.csv", index_col=0)["cluster"]
gap_top = pd.read_csv(reports_dir / "gap_vs_top_group.csv", index_col=0)
ranked = index[index["Rank"].notna()].index

AREA = {
    "ger": "Education", "life_expectancy": "Health", "unemployment_rate": "Jobs",
    "cd_ratio": "Finance", "per_capita_power": "Power supply", "td_losses": "Power losses",
    "road_density": "Roads", "factory_density": "Industry", "msme_density": "MSMEs",
    "manufacturing_share": "Manufacturing", "investment_rate": "Investment",
}

print("states:", len(ranked), "| groups:", clusters.nunique(), "| gap table:", gap_top.shape)

states: 33 | groups: 3 | gap table: (33, 11)


## Priorities by state type

For each group, count the numbers that most often appear among its states' two biggest gaps
to the top group. This shows the shared priority areas for each type.

In [6]:
from collections import Counter

def top_gaps(state, n=2):
    g = gap_top.loc[state]
    return list(g[g > 0].sort_values(ascending=False).index[:n])

order = ["Strong on both parts", "Strong basic conditions, weaker industry", "Weak on both parts"]
for cl in order:
    members = clusters[clusters == cl].index
    counts = Counter()
    for s in members:
        counts.update(top_gaps(s))
    print(f"{cl} ({len(members)} states):")
    for ind, c in counts.most_common(4):
        print(f"    {ind} ({AREA[ind]}): {c}")
    print()

Strong on both parts (12 states):
    cd_ratio (Finance): 4
    investment_rate (Investment): 4
    manufacturing_share (Manufacturing): 4
    msme_density (MSMEs): 3

Strong basic conditions, weaker industry (6 states):
    investment_rate (Investment): 6
    manufacturing_share (Manufacturing): 4
    factory_density (Industry): 2

Weak on both parts (15 states):
    factory_density (Industry): 11
    manufacturing_share (Manufacturing): 6
    investment_rate (Investment): 5
    ger (Education): 3



## Priorities by state

For each state: its main priority (biggest gap to the top group), its secondary priority
(second biggest gap), and its smallest gap. Shown as broad areas.

In [7]:
def easiest_gap(state):
    g = gap_top.loc[state]
    below = g[g > 0]
    return below.idxmin() if len(below) else None

rows = []
for s in ranked:
    gaps = top_gaps(s, n=2)
    main = gaps[0] if len(gaps) > 0 else None
    secondary = gaps[1] if len(gaps) > 1 else None
    easy = easiest_gap(s)
    rows.append({
        "rank": int(index.loc[s, "Rank"]),
        "main_priority": AREA[main] if main else "-",
        "secondary_priority": AREA[secondary] if secondary else "-",
        "smallest_gap": AREA[easy] if easy else "-",
    })
priorities = pd.DataFrame(rows, index=ranked).sort_values("rank")
priorities.to_csv(reports_dir / "development_priorities.csv", index_label="state")

print("saved:", reports_dir / "development_priorities.csv")
print(priorities.to_string())

saved: D:\india-state-competitiveness-index\version2\reports\development_priorities.csv
                           rank  main_priority secondary_priority   smallest_gap
state                                                                           
Goa                           1           Jobs            Finance        Finance
Tamil Nadu                    2     Investment      Manufacturing   Power losses
Gujarat                       3      Education             Health          Roads
Puducherry                    4          Roads              MSMEs        Finance
Telangana                     5  Manufacturing         Investment      Education
Andhra Pradesh                6  Manufacturing         Investment         Health
Punjab                        7  Manufacturing            Finance          Roads
Maharashtra                   8       Industry         Investment          Roads
Himachal Pradesh              9          MSMEs            Finance   Power supply
Haryana              

## Cross-state findings

Count how often each area is a state's main priority.

In [8]:
print("main priority areas across states:")
print(priorities["main_priority"].value_counts().to_string())

industry_areas = {"Industry", "Manufacturing", "Investment", "MSMEs"}
n_industry = priorities["main_priority"].isin(industry_areas).sum()
print(f"\nstates whose main priority is an industry-related area: {n_industry} of {len(priorities)}")

main priority areas across states:
main_priority
Industry         11
Investment        7
Manufacturing     6
MSMEs             2
Health            2
Jobs              1
Education         1
Roads             1
Finance           1
Power losses      1

states whose main priority is an industry-related area: 26 of 33
